# LARA Training — Google Colab
Modifie `EXPERIMENT` ci-dessous puis exécute toutes les cellules.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────
EXPERIMENT   = "diff_attn"  # diff_attn | mor | coconut | lara_full | lara_v2_full | lara_v2_dca
MAX_ITERS    = 5000
BATCH_SIZE   = 8
GRAD_ACCUM   = 16
N_EMBD       = 1024
N_LAYER      = 6
N_HEAD       = 8
BLOCK_SIZE   = 512
LR           = "3e-4"
WARMUP       = 500
N_RECURSIONS = 4
WANDB        = False
# ──────────────────────────────────────────────────────────────

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/LARA_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints : {DRIVE_DIR}')

In [ ]:
import os

# Cloner ou mettre à jour le repo
if os.path.exists('/content/LARA'):
    !git -C /content/LARA pull
else:
    !git clone https://github.com/s3basti3nDev/LARA.git /content/LARA

# Trouver le bon dossier (lara/ ou racine)
if os.path.exists('/content/LARA/lara/train.py'):
    LARA_DIR = '/content/LARA/lara'
elif os.path.exists('/content/LARA/train.py'):
    LARA_DIR = '/content/LARA'
else:
    raise FileNotFoundError('train.py introuvable — vérifie que le repo GitHub contient bien les fichiers lara/')

os.chdir(LARA_DIR)
print(f'Dossier de travail : {LARA_DIR}')
!ls

In [ ]:
# Installer les dépendances (torch déjà présent dans Colab)
!pip install tiktoken datasets wandb -q
!nvidia-smi | grep -E 'GPU|Memory'

In [ ]:
import subprocess, sys, os

cmd = [
    sys.executable, 'train.py',
    '--experiment',    EXPERIMENT,
    '--dataset',       'fineweb',
    '--n_embd',        str(N_EMBD),
    '--n_layer',       str(N_LAYER),
    '--n_head',        str(N_HEAD),
    '--block_size',    str(BLOCK_SIZE),
    '--learning_rate', LR,
    '--warmup_iters',  str(WARMUP),
    '--batch_size',    str(BATCH_SIZE),
    '--grad_accum',    str(GRAD_ACCUM),
    '--max_iters',     str(MAX_ITERS),
    '--device',        'cuda',
    '--out_dir',       DRIVE_DIR,
]
if EXPERIMENT in ('lara_v2', 'lara_v2_dca', 'lara_v2_full'):
    cmd += ['--n_recursions', str(N_RECURSIONS)]
if WANDB:
    cmd.append('--wandb')

print('Lancement :', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Evaluer le checkpoint
import subprocess, sys
ckpt = f'{DRIVE_DIR}/exp_{EXPERIMENT}_best.pt'
subprocess.run([sys.executable, 'evaluate.py', '--model', ckpt, '--dataset', 'fineweb'], check=True)